# Jev sur CPU — taille d'entrée, candidats et débit

Choisir le kernel **Minecraft experiments - Jev 0.8B**. Ce notebook utilise
uniquement le CPU et les poids déjà téléchargés. Aucun serveur Minecraft n'est connecté.

Les résultats mesurés sont dans `runs/cpu_benchmark_latest.json`.
On peut les lire directement ou refaire les mesures en changeant `RUN_BENCHMARK`.
Le chargement initial est séparé des appels ; le modèle reste chargé.

Une **requête** peut contenir plusieurs questions. Ici chaque requête contient une question.
`choice` avec K actions = K entrées candidates dans un batch ; `noul` = une seule entrée.
Les tokens de l'état sont répétés pour chaque candidat. La limite du checkpoint est
**768 tokens par candidat formaté**, pas 768 tokens pour l'ensemble des candidats.
[Fiche du modèle](https://huggingface.co/guoan1/jev-qwen35-08b).

In [ ]:
import json
import sys
from pathlib import Path
from IPython.display import HTML, display

ROOT = Path.cwd()
if not (ROOT / "jev_sandbox.py").is_file():
    ROOT = ROOT / "experiments"
if not (ROOT / "jev_sandbox.py").is_file():
    raise RuntimeError("Ouvrir le notebook depuis experiments.")
if Path(sys.executable).resolve() != (ROOT / ".venv" / "Scripts" / "python.exe").resolve():
    raise RuntimeError("Choisir le kernel Minecraft experiments - Jev 0.8B.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import torch
from jev_sandbox import JevSandbox
from jev_experiments import html_table, machine_info, request_profile

# Le GPU n'est pas utilisé, même s'il est disponible.
lab = JevSandbox(device="cpu")
CPU_THREADS = 8  # Réglage le plus rapide mesuré ici parmi 1, 2, 4, 8
torch.set_num_threads(CPU_THREADS)
print(json.dumps(machine_info(), indent=2, ensure_ascii=False))
print(f"Chargement : {lab.load_seconds:.3f} s ; limite : {lab.max_length} tokens par candidat")

## Voir la taille réelle d'une décision

`request_profile` applique exactement le format de l'auteur et le chat template,
sans appeler le modèle. `input_tokens` compte les vrais tokens ; `padded_tokens`
inclut l'alignement des candidats du batch. Les deux peuvent influencer le coût.

In [ ]:
from jev_experiments import make_request
state, questions, profile = make_request(lab, token_budget=256, candidates=4)
print(json.dumps(profile, indent=2))

## Mesures enregistrées ou nouvelle exécution

Les entrées longues contiennent du remplissage neutre : on mesure la latence,
pas la qualité. Chaque cas est chauffé une fois, puis chronométré plusieurs fois.
Les requêtes sont séquentielles, sans trafic concurrent. L'appel chronométré inclut
compilation, tokenisation, forward et formatage de réponse ; il exclut le chargement.

Le débit `requêtes/s = 1000 / médiane_ms` est un débit séquentiel observé dans ce test,
pas un plafond de concurrence. `candidats/s` compte les entrées scorées, pas des
tokens générés. Cinq répétitions donnent une indication, pas un percentile fiable.

In [ ]:
from jev_experiments import run_cpu_benchmark, save_benchmark

RUN_BENCHMARK = False  # True pour refaire la matrice sur ton CPU
TOKEN_BUDGETS = [128, 256, 512, 768]
CANDIDATE_COUNTS = [2, 4, 8]
REPETITIONS = 5

saved = ROOT / "runs" / "cpu_benchmark_latest.json"
if RUN_BENCHMARK or not saved.exists():
    results = run_cpu_benchmark(lab, TOKEN_BUDGETS, CANDIDATE_COUNTS, REPETITIONS)
    print("Enregistré :", save_benchmark(results))
else:
    results = json.loads(saved.read_text(encoding="utf-8"))
    print("Mesures enregistrées :", results["machine"]["local_time"])
    print("Threads lors de ces mesures :", results["machine"]["threads"])

display(HTML(html_table(results["rows"], [
    ("type", "Type"), ("candidate_prompts", "Candidats"),
    ("longest_prompt_tokens", "Tokens / candidat max"), ("input_tokens", "Tokens totaux"),
    ("median_ms", "Médiane ms"), ("min_ms", "Min ms"), ("max_ms", "Max ms"),
    ("requests_per_second", "Requêtes / s"),
    ("candidate_prompts_per_second", "Candidats / s"),
])))
print(json.dumps(results["machine"], indent=2, ensure_ascii=False))

## Effet du nombre de threads

Le nombre de threads peut peser autant que la taille du texte. Cette comparaison
utilise un seul cas (4 candidats, 256 tokens) ; le meilleur réglage peut changer
sur les entrées longues. Les résultats de la matrice gardent un réglage fixe.

In [ ]:
from jev_experiments import thread_sweep
RUN_THREAD_SWEEP = False
thread_rows = thread_sweep(lab) if RUN_THREAD_SWEEP else results.get("thread_sweep", [])
display(HTML(html_table(thread_rows, [("threads", "Threads"), ("median_ms", "Médiane ms"),
                                     ("requests_per_second", "Requêtes / s")])))

## Dépasser la limite

La limite de 768 vient de cette variante Jev et de son runtime. Le contexte annoncé
pour le Qwen de base ne permet pas de supposer que cet adaptateur fonctionne bien
sur des états plus longs. On ne relève pas la limite pour contourner le test.

In [ ]:
too_long_state, too_long_questions, too_long_profile = make_request(lab, 800, 2)
print("Longueur réelle :", too_long_profile["longest_prompt_tokens"])
try:
    lab.decide(too_long_state, too_long_questions)
except ValueError as error:
    print("Refus attendu avant le forward :", error)
else:
    raise AssertionError("Le runtime aurait dû refuser l'entrée trop longue.")

## Tester ton propre état

Modifier l'état et les descriptions. Retirer les informations inutiles peut réduire
la latence, mais aussi changer la décision. Il faut mesurer les deux séparément.

In [ ]:
from jev_experiments import measure_request
my_state = {"task": "Collect 10 logs", "inventory_logs": 4,
            "selected_tree_distance": 1, "nearby_dropped_log_items": 2}
my_questions = {"next": {"type": "choice", "instructions": "Choose the next action.",
                        "criteria": {"chop": "Break a reachable log block.",
                                     "pickup": "Pick up nearby dropped logs.",
                                     "stop": "The inventory goal is already complete."}}}
print(json.dumps(request_profile(lab, my_state, my_questions), indent=2))
print(json.dumps(measure_request(lab, my_state, my_questions, repetitions=3), indent=2))

Exécuter la cellule suivante quand tu as terminé pour libérer le modèle.

In [ ]:
lab.close()